In [1]:
import os 
import pandas as pd 
import json
import numpy as np
initial_dir = os.getcwd()
os.chdir('/Users/valerio/Documents/Borelli/These/Subdimensional Motifs /')
from multivariate_tsmd.new_multivariate_synthetic_signal import NewMultivariateSignalGenerator, CoOccurringSignalGenerator
from multivariate_tsmd.utils import plot_signal_and_motifs, plot_signal_and_submotifs
os.chdir(initial_dir)
import random
random.seed(42)

In [2]:
synthetic_path = os.getcwd() +'/synthetic'

In [3]:
import os
import json
import pandas as pd
import numpy as np

def generate_scenario_dataset(
    synthetic_path,
    scenario_name,
    variable_param_name,
    variable_param_values,
    generator_params,
    n_signals_per_setting=10,
    save_signals_as="csv"
):
    """
    Génère un dataset de scénarios synthétiques pour différentes valeurs de paramètres.

    Args:
        synthetic_path (str): Dossier racine où enregistrer le scénario.
        scenario_name (str): Nom du sous-dossier de scénario (ex: "ratio_active_dimensions").
        variable_param_name (str): Nom du paramètre à faire varier (ex: "n_actives_dimensions_ratio").
        variable_param_values (list): Liste des valeurs à tester pour ce paramètre.
        generator_params (dict): Paramètres du générateur de signaux.
        n_signals_per_setting (int): Nombre de signaux à générer par valeur de paramètre.
        save_signals_as (str): Format de sauvegarde du signal ("csv" ou "npy").
    """

    scenario_path = os.path.join(synthetic_path, scenario_name)
    os.makedirs(scenario_path, exist_ok=True)

    for val in variable_param_values:
        print(f"=== Génération pour {variable_param_name} = {val} ===")

        # Mettre à jour le paramètre variable
        generator_params[variable_param_name] = val

        # Dossier spécifique à cette valeur
        subfolder = os.path.join(scenario_path, f"{variable_param_name}_{val}","Data")
        os.makedirs(subfolder, exist_ok=True)

        for i in range(n_signals_per_setting):
            # Génération du signal
            signal_gen = NewMultivariateSignalGenerator(**generator_params)
            signal, labels = signal_gen.generate()

            # Conversion signal → DataFrame pour export CSV
            df_signal = pd.DataFrame(signal)

            # Conversion des structures complexes pour JSON
            positions = {}
            for k, v in signal_gen.positions_.items():
                motif_list=[]
                for u in v:
                    motif_list.append([u[0], u[0]+u[1]])
                positions[str(k)] = motif_list

            active_dims_dict = {str(k): v if type(v) == list else v.tolist() for k, v in enumerate(signal_gen.active_dimensions)}

            # Définition des chemins
            base_name = f"signal_{i:02d}"
            signal_path = os.path.join(subfolder, f"{base_name}.{save_signals_as}")
            json_path = os.path.join(subfolder, f"metadata_{i:02d}.json")

            # Sauvegarde du signal
            if save_signals_as == "csv":
                df_signal.to_csv(signal_path, index=False)
            elif save_signals_as == "npy":
                np.save(signal_path.replace(".npy", ""), signal)
            else:
                raise ValueError("Format non supporté. Utiliser 'csv' ou 'npy'.")

            # Sauvegarde du metadata JSON
            metadata = {
                "occurences_positions": positions,
                "active_dims": active_dims_dict,
                "generator_params": generator_params
            }

            with open(json_path, "w") as f:
                json.dump(metadata, f, indent=4)

        #print(f"→ {n_signals_per_setting} signaux générés dans {subfolder}\n")

    print("✅ Génération terminée.")


## Number of motifs

In [ ]:
n_motifs=[1,2,5,10,15,20,50]
generator_params={'n_d':5, 'n_actives_dimensions_ratio':0.7,'motif_length':50, 'motif_amplitude' :4 , 'motif_fundamental':4,
                   'sparsity':0.8,'sparsity_fluctuation':0.5,'length_fluctuation':0,'min_rep':4,'max_rep':4,'noise_amplitude':0.1}
scenario_name='number_of_motifs'
variable_param_name='n_motifs'
variable_param_values=n_motifs

generate_scenario_dataset(
    synthetic_path,
    scenario_name,
    variable_param_name,
    variable_param_values,
    generator_params,
    n_signals_per_setting=10,
    save_signals_as="csv"
)

=== Génération pour n_motifs = 1 ===
=== Génération pour n_motifs = 2 ===
=== Génération pour n_motifs = 5 ===
=== Génération pour n_motifs = 10 ===
=== Génération pour n_motifs = 15 ===
=== Génération pour n_motifs = 20 ===
=== Génération pour n_motifs = 50 ===
✅ Génération terminée.
